# Eksperimen SML - Farrel Ghozy Affifudin

## 1. Perkenalan Dataset

**Dataset**: Pima Indians Diabetes Database

**Sumber**: Kaggle / UCI Machine Learning Repository

Dataset ini berasal dari National Institute of Diabetes and Digestive and Kidney Diseases. Tujuan dari dataset ini adalah untuk memprediksi secara diagnostis apakah seorang pasien menderita diabetes atau tidak, berdasarkan pengukuran diagnostik tertentu yang terdapat dalam dataset. Beberapa batasan yang perlu diperhatikan adalah, seluruh pasien yang tercatat dalam dataset ini adalah wanita berusia minimal 21 tahun keturunan Indian Pima.

### Informasi Fitur:
| Fitur | Tipe | Deskripsi |
|-------|------|-----------|
| Pregnancies | Numerik | Jumlah kehamilan |
| Glucose | Numerik | Konsentrasi glukosa plasma (mg/dL) |
| BloodPressure | Numerik | Tekanan darah diastolik (mm Hg) |
| SkinThickness | Numerik | Ketebalan lipatan kulit trisep (mm) |
| Insulin | Numerik | Insulin serum 2 jam (mu U/mL) |
| BMI | Numerik | Indeks massa tubuh (kg/m²) |
| DiabetesPedigreeFunction | Numerik | Fungsi silsilah diabetes |
| Age | Numerik | Usia (tahun) |
| Outcome | Kategorikal | Hasil diagnosis (0 = Non-Diabetic, 1 = Diabetic) |

**Tujuan**: Membangun model klasifikasi untuk memprediksi kemungkinan seseorang menderita diabetes berdasarkan fitur-fitur medis yang tersedia.

## 2. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
print("[INFO] Semua library berhasil diimpor.")

## 3. Memuat Dataset

Pada tahap ini, dataset dimuat ke dalam notebook menggunakan pandas. Dataset yang digunakan adalah Pima Indians Diabetes dalam format CSV.

In [ ]:
df = pd.read_csv('../diabetes_raw/diabetes_with_headers.csv')
print(f"[INFO] Dataset berhasil dimuat.")
print(f"[INFO] Shape: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"[INFO] Nama kolom: {list(df.columns)}")
print()
df.head(10)

## 4. Exploratory Data Analysis (EDA)

EDA dilakukan untuk memahami karakteristik dataset, mendeteksi anomali, dan menentukan langkah preprocessing yang tepat.

### 4.1 Informasi Dataset

In [ ]:
print("=== Info Dataset ===")
print(df.info())
print(f"\nDuplicated rows: {df.duplicated().sum()}")
print(f"\nMissing values:\n{df.isnull().sum()}")

### 4.2 Statistik Deskriptif

In [ ]:
print("=== Statistik Deskriptif ===")
df.describe().T

### 4.3 Distribusi Target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
target_counts = df['Outcome'].value_counts().sort_index()
colors = ['#2ecc71', '#e74c3c']
labels = ['Non-Diabetic (0)', 'Diabetic (1)']

axes[0].bar(labels, target_counts.values, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Distribusi Target', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Jumlah', fontsize=12)
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

axes[1].pie(target_counts.values, labels=labels, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0.02, 0.02))
axes[1].set_title('Proporsi Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('preprocessing/diabetes_preprocessing/target_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

### 4.4 Deteksi Nilai Nol yang Tidak Valid

In [ ]:
columns_with_zero = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
zero_counts = {col: (df[col] == 0).sum() for col in columns_with_zero}
zero_df = pd.DataFrame(list(zero_counts.items()), columns=['Fitur', 'Jumlah Nol'])
zero_df['Persentase'] = (zero_df['Jumlah Nol'] / len(df) * 100).round(2)
print(zero_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(zero_df['Fitur'], zero_df['Jumlah Nol'], color='#e74c3c', edgecolor='black')
ax.set_title('Jumlah Nilai Nol per Fitur', fontsize=14, fontweight='bold')
ax.set_ylabel('Jumlah', fontsize=12)
for bar, val in zip(bars, zero_df['Jumlah Nol']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(val), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('preprocessing/diabetes_preprocessing/zero_values.png', dpi=100, bbox_inches='tight')
plt.show()

### 4.5 Analisis Korelasi

In [ ]:
plt.figure(figsize=(10, 8))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Heatmap Korelasi Antar Fitur', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('preprocessing/diabetes_preprocessing/correlation_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

### 4.6 Distribusi Setiap Fitur

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
feature_cols = [c for c in df.columns if c != 'Outcome']
axes = axes.flatten()
for i, col in enumerate(feature_cols):
    axes[i].hist(df[col], bins=30, color='#3498db', edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Distribusi {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col, fontsize=10)
    axes[i].set_ylabel('Frekuensi', fontsize=10)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.savefig('preprocessing/diabetes_preprocessing/feature_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Data Preprocessing

Preprocessing dilakukan untuk membersihkan dan mempersiapkan data sebelum digunakan dalam model machine learning.

### 5.1 Menangani Missing Values (Nilai Nol yang Tidak Valid)

In [ ]:
df_clean = df.copy()
columns_to_replace = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in columns_to_replace:
    median_val = df_clean[df_clean[col] != 0][col].median()
    df_clean[col] = df_clean[col].replace(0, median_val)
print("[INFO] Nilai nol pada fitur medis telah diganti dengan median.")
print(f"\nCek nilai nol setelah preprocessing:")
print((df_clean[columns_to_replace] == 0).sum())

### 5.2 Feature Scaling (StandardScaler)

In [ ]:
scaler = StandardScaler()
feature_cols = [c for c in df_clean.columns if c != 'Outcome']
df_scaled = df_clean.copy()
df_scaled[feature_cols] = scaler.fit_transform(df_clean[feature_cols])
print("[INFO] StandardScaler telah diterapkan.")
print(f"\nStatistik setelah scaling:")
df_scaled[feature_cols].describe().T

### 5.3 Split Data Train-Test

In [ ]:
X = df_scaled[feature_cols]
y = df_scaled['Outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"[INFO] Split data: Train={X_train.shape[0]}, Test={X_test.shape[0]}")
print(f"\nDistribusi target train:")
print(y_train.value_counts(normalize=True).mul(100).round(2))
print(f"\nDistribusi target test:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

### 5.4 Menyimpan Hasil Preprocessing

In [ ]:
output_dir = 'preprocessing/diabetes_preprocessing'
os.makedirs(output_dir, exist_ok=True)

train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)
train_df.to_csv(f'{output_dir}/train.csv', index=False)
test_df.to_csv(f'{output_dir}/test.csv', index=False)
df_clean.to_csv(f'{output_dir}/diabetes_clean.csv', index=False)
joblib.dump(scaler, f'{output_dir}/scaler.pkl')

print("[INFO] Hasil preprocessing disimpan:")
print(f"  - {output_dir}/train.csv")
print(f"  - {output_dir}/test.csv")
print(f"  - {output_dir}/diabetes_clean.csv")
print(f"  - {output_dir}/scaler.pkl")

### 5.5 Summary

In [ ]:
print("=" * 60)
print("EKSPERIMEN SELESAI")
print("=" * 60)
print(f"\nDataset: Pima Indians Diabetes")
print(f"Total sampel: {len(df)}")
print(f"Fitur: {len(feature_cols)}")
print(f"Train set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"\nFitur yang digunakan: {feature_cols}")
print(f"\nTarget encoding: 0 = Non-Diabetic, 1 = Diabetic")
print("\nDataset siap digunakan untuk pemodelan.")